# DataLineageML v0.2 — Oyo State Agricultural Subsidy Bias Case Study

**Scenario:** A crop yield model allocates fertiliser subsidies to smallholder farmers in Oyo State, Nigeria. An audit finds female-headed farms are 34% less likely to receive subsidies than male-headed farms with equivalent yields.

**This notebook demonstrates the complete v0.2 API:**

| Step | API used |
|---|---|
| 0 | `discover_sensitive_cols()` — auto-detect demographic columns |
| 1 | `@track(snapshot=True)` + `LineageContext` — instrument pipeline |
| 2 | `DataFrameProfiler` — inspect demographic distributions |
| 3 | `ShiftDetector` — rank steps by JSD + KS shift |
| 4 | `CausalAttributor` — identify the causal step |
| 5 | `DemographicParityGap` — compute bias metric (no model needed) |
| 6 | `EqualizedOdds` + `PredictiveParity` — post-model fairness metrics |
| 7 | `CounterfactualReplayer` — verify the fix with `register_tracked()` |
| 8 | `CrossRunComparator` — detect drift across multiple pipeline runs |
| 9 | `generate_report()` — export self-contained HTML audit report |

> All data in this notebook is **synthetic**, generated to reflect documented structural inequalities in Oyo State agricultural registration. No real farmer records are used.

## 0. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../src'))  # local dev

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import datalineageml as dlm
from datalineageml import track, LineageContext, LineageStore, LineageGraph
from datalineageml import CounterfactualReplayer, generate_report
from datalineageml.analysis import (
    DataFrameProfiler,
    ShiftDetector,
    CausalAttributor,
    CrossRunComparator,
    DemographicParityGap,
    EqualizedOdds,
    PredictiveParity,
    RegressionFairnessAuditor,
    discover_sensitive_cols,
    suggest_sensitive_cols,
    print_sensitive_col_report,
    print_snapshot_comparison,
)

print(f'DataLineageML version: {dlm.__version__}')
print('All imports OK.')

DataLineageML version: 0.2.0
All imports OK.


## 1. Synthetic Oyo State Data

Structural inequalities baked into the data (reflecting real registration gaps):

| Field | Female access | Male access |
|---|---|---|
| Formal land title | 11% | 67% |
| Registered mobile | 55% | 88% |
| GPS coordinates logged | 48% | 82% |

These gaps are properties of the registration system, not of farming performance. The bias in the model is an artefact of data collection, not of agricultural reality.

In [2]:
np.random.seed(42)
N = 500

gender    = np.random.choice(['F', 'M'], N, p=[0.40, 0.60])
is_f      = gender == 'F'

farm_size = np.where(is_f,
    np.random.exponential(0.8, N).clip(0.2, 5.0),
    np.random.exponential(1.4, N).clip(0.3, 10.0))

# Ground truth: equal yield per hectare — no gender gap in productivity
crop_yield = (1.8 + 0.4 * farm_size + np.random.normal(0, 0.3, N)).clip(0.5, 5.0)

# Land title — correlated with gender (the key missing field)
land_title = np.where(is_f,
    np.random.choice(['registered', None], N, p=[0.11, 0.89]),
    np.random.choice(['registered', None], N, p=[0.67, 0.33])).tolist()

# Mobile number
mobile_prob = np.where(is_f, 0.55, 0.88)
mobile = np.where(np.random.binomial(1, mobile_prob),
    [f'0{np.random.randint(700,900):03d}{np.random.randint(1000000,9999999)}' for _ in range(N)],
    None).tolist()

# GPS
gps_prob  = np.where(is_f, 0.48, 0.82)
has_gps   = np.random.binomial(1, gps_prob)
latitude  = np.where(has_gps, np.random.uniform(7.2, 8.8, N), np.nan)
longitude = np.where(has_gps, np.random.uniform(3.2, 4.5, N), np.nan)

soil_quality = np.random.uniform(3, 9, N)
experience   = np.random.randint(1, 30, N)

# Ground truth: gender-neutral eligibility criterion
subsidy_eligible = ((crop_yield > 2.0) & (farm_size > 0.5)).astype(int)

df_raw = pd.DataFrame({
    'gender':           gender,
    'farm_size_ha':     farm_size.round(2),
    'crop_yield_t_ha':  crop_yield.round(3),
    'soil_quality':     soil_quality.round(2),
    'experience_yrs':   experience,
    'land_title':       land_title,
    'mobile_number':    mobile,
    'latitude':         latitude.round(6),
    'longitude':        longitude.round(6),
    'subsidy_eligible': subsidy_eligible,
})

print(f'Dataset: {df_raw.shape[0]} farms, {df_raw.shape[1]} columns')
print(f'Gender: {df_raw.gender.value_counts().to_dict()}')
print(f'\nNull counts:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0].to_string())
print(f'\nGround truth eligibility by gender (should be similar):')
print(df_raw.groupby('gender')['subsidy_eligible'].mean().round(3).to_string())

Dataset: 500 farms, 10 columns
Gender: {'M': 293, 'F': 207}

Null counts:
land_title       295
mobile_number    127
latitude         160
longitude        160

Ground truth eligibility by gender (should be similar):
gender
F    0.386
M    0.628


## 2. Auto-Discover Sensitive Columns

Before instrumenting the pipeline, use `discover_sensitive_cols()` to automatically identify which columns are likely demographic attributes — no manual specification needed.

In [3]:
# Auto-discover sensitive columns
print_sensitive_col_report(df_raw, min_confidence=0.5)

# Get just the column names to use in @track decorators
sensitive_cols = suggest_sensitive_cols(df_raw, min_confidence=0.6)
print(f'Suggested for tracking: sensitive_cols={sensitive_cols}')


─────────────────────────────────────────────────────────────────
  Sensitive column discovery
─────────────────────────────────────────────────────────────────
  Column                 Conf  Unique     Dtype  Top values
  ────────────────────  ─────  ──────  ────────  ────────────────────────
  gender                 1.00       2    object  M, F

  Suggested: sensitive_cols=["gender"]
─────────────────────────────────────────────────────────────────

Suggested for tracking: sensitive_cols=['gender']


## 3. Profile Raw Data

Before running the pipeline, use `DataFrameProfiler` to inspect the distribution of sensitive attributes in the raw data.

In [4]:
profiler = DataFrameProfiler(sensitive_cols=['gender'])
profiler.print_profile(df_raw, step_name='raw_data')


─────────────────────────────────────────────────────────────────
  Profile: raw_data
─────────────────────────────────────────────────────────────────
  Rows: 500   Columns: 10

  Null rates (columns with missing data):
    land_title                 ████████████░░░░░░░░  59.0%
    latitude                   ██████░░░░░░░░░░░░░░  32.0%
    longitude                  ██████░░░░░░░░░░░░░░  32.0%
    mobile_number              █████░░░░░░░░░░░░░░░  25.4%

  Numeric features (mean ± std):
    farm_size_ha                    1.213 ± 1.302   [0.20, 10.00]
    crop_yield_t_ha                 2.300 ± 0.599   [1.02, 5.00]
    soil_quality                    5.967 ± 1.735   [3.02, 9.00]
    experience_yrs                 14.850 ± 8.494   [1.00, 29.00]
    latitude                        8.011 ± 0.465   [7.20, 8.80]
    longitude                       3.849 ± 0.382   [3.21, 4.50]
    subsidy_eligible                0.528 ± 0.500   [0.00, 1.00]

  Sensitive column distributions:
    gender:
    

## 4. Instrument and Run the Biased Pipeline

Every step is decorated with `@track`. The `clean_data` step uses `snapshot=True` — demographic distributions are logged automatically before and after.

In [5]:
# Clean up any previous runs
for f in ['oyo_lineage_biased.db', 'oyo_lineage_fixed.db']:
    if os.path.exists(f): os.remove(f)

dlm.init(db_path='oyo_lineage_biased.db')
STORE = dlm.get_default_store()
print('Store initialised → oyo_lineage_biased.db')

Store initialised → oyo_lineage_biased.db


In [6]:
@track(name='load_farm_data', tags={'source': 'oyo_registry_2025', 'stage': 'ingestion'})
def load_farm_data(df):
    return df.copy()


# snapshot=True + sensitive_cols logs gender distribution before AND after
@track(
    name='clean_data',
    tags={'stage': 'preprocessing'},
    snapshot=True,
    sensitive_cols=['gender'],   # auto-discovered above
)
def clean_data(df):
    """Biased: dropna removes female farmers disproportionately."""
    return df.dropna().reset_index(drop=True)


@track(name='engineer_features', tags={'stage': 'feature_engineering'})
def engineer_features(df):
    df = df.copy()
    df['yield_per_ha_score'] = df['crop_yield_t_ha'] / df['farm_size_ha']
    df['soil_yield_index']   = df['soil_quality'] * df['crop_yield_t_ha'] / 10
    df['exp_category']       = pd.cut(df['experience_yrs'],
                                      bins=[0,5,15,35],
                                      labels=['beginner','intermediate','veteran'])
    return df


@track(name='encode_categoricals', tags={'stage': 'preprocessing'})
def encode_categoricals(df):
    df = df.copy()
    df['land_title_enc'] = (df['land_title'] == 'registered').astype(int)
    df['exp_category_enc'] = df['exp_category'].cat.codes
    return df


@track(name='normalize_features', tags={'stage': 'preprocessing'})
def normalize_features(df):
    df = df.copy()
    for c in ['farm_size_ha', 'crop_yield_t_ha', 'soil_quality',
               'experience_yrs', 'yield_per_ha_score', 'soil_yield_index']:
        lo, hi = df[c].min(), df[c].max()
        if hi > lo: df[c] = (df[c] - lo) / (hi - lo)
    return df


@track(name='train_subsidy_model', tags={'stage': 'training', 'model': 'random_forest'})
def train_subsidy_model(df):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score
    features = ['farm_size_ha','crop_yield_t_ha','soil_quality','experience_yrs',
                'yield_per_ha_score','soil_yield_index','land_title_enc','exp_category_enc']
    X = df[features]; y = df['subsidy_eligible']
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    cv    = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    model.fit(X, y)
    print(f'  CV accuracy: {cv.mean():.3f} ± {cv.std():.3f}')
    return model, features


print('Pipeline functions defined.')

Pipeline functions defined.


In [7]:
print('Running BIASED pipeline...\n')
with LineageContext(name='oyo_subsidy_pipeline_v1_biased'):
    d1 = load_farm_data(df_raw)
    d2 = clean_data(d1)              # ← bias introduced here
    d3 = engineer_features(d2)
    d4 = encode_categoricals(d3)
    d5 = normalize_features(d4)
    model_biased, features = train_subsidy_model(d5)

print(f'\nRows after clean_data: {len(d2)} / {len(d1)} ({len(d2)/len(d1):.1%} retained)')

Running BIASED pipeline...

  CV accuracy: 0.986 ± 0.017

Rows after clean_data: 139 / 500 (27.8% retained)


## 5. Detect Distribution Shifts (ShiftDetector)

`ShiftDetector` reads the logged snapshots and ranks every step by demographic shift magnitude using Jensen-Shannon divergence (categorical) and the KS statistic (numeric).

In [8]:
detector = ShiftDetector(store=STORE)
shifts   = detector.detect()
detector.print_report(shifts, title='Shift Report — oyo_subsidy_pipeline_v1_biased')


════════════════════════════════════════════════════════════════════════
  Shift Report — oyo_subsidy_pipeline_v1_biased
════════════════════════════════════════════════════════════════════════
  Steps analysed: 1   Signals: 8   HIGH: 1   MEDIUM: 3   LOW: 4
  JSD thresholds: HIGH ≥ 0.020  MEDIUM ≥ 0.005   KS thresholds: HIGH ≥ 0.20  MEDIUM ≥ 0.10

  #    Step                     Column         Test     Stat  Flag          Rows removed
  ──── ──────────────────────── ────────────── ───── ───────  ──────────  ──────────────
  1    clean_data               gender         jsd    0.1681  ⚠ HIGH           361 (72.2%)
  2    clean_data               farm_size_ha   ks     0.1378  △ MEDIUM         361 (72.2%)
  3    clean_data               subsidy_eligible ks     0.1113  △ MEDIUM         361 (72.2%)
  4    clean_data               crop_yield_t_ha ks     0.1017  △ MEDIUM         361 (72.2%)
  5    clean_data               longitude      ks     0.0519    LOW            361 (72.2%)
  6    clean_

In [9]:
# Manual before/after comparison at clean_data
snaps  = STORE.get_snapshots('clean_data')
before = next(s for s in snaps if s['position'] == 'before')
after  = next(s for s in snaps if s['position'] == 'after')
print_snapshot_comparison(before, after, step_name='clean_data')


═════════════════════════════════════════════════════════════════
  Snapshot comparison: clean_data
═════════════════════════════════════════════════════════════════
                                 BEFORE       AFTER     SHIFT
  Rows                              500         139      -361

  gender distribution:
    F                   █████░░░░░░░ 41.4%  →  ░░░░░░░░░░░░  3.6%  -37.8%  ⚠  HIGH SHIFT
    M                   ███████░░░░░ 58.6%  →  ████████████ 96.4%  +37.8%  ⚠  HIGH SHIFT

  ⚠  Significant demographic shift detected at 'gender'.
     This step is a candidate causal source of bias.

═════════════════════════════════════════════════════════════════



## 6. Attribute the Causal Step (CausalAttributor)

In [10]:
attributor  = CausalAttributor(store=STORE)
attribution = attributor.attribute(sensitive_col='gender')
attributor.print_attribution(attribution)

attributed_step = attribution['attributed_step']
print(f'\nAttribted step: {attributed_step}  (confidence {attribution["confidence"]:.0%})')


════════════════════════════════════════════════════════════════════════
  DataLineageML — Causal Attribution Report
════════════════════════════════════════════════════════════════════════
  Attributed step:  clean_data
  Sensitive column: gender
  Shift stat:       0.1681 (JSD)  [⚠ HIGH]
  Confidence:       100.0%
  Rows removed:     361  (72.2% of dataset)

────────────────────────────────────────────────────────────────────────
  Evidence
────────────────────────────────────────────────────────────────────────
  The 'gender' distribution shifted high at 'clean_data' (JSD = 0.1681). 361 rows were removed (72.2% of the dataset) at this step.

  'gender' distribution at 'clean_data':
    F                 █████░░░░░░░ 41.4%  →  ░░░░░░░░░░░░  3.6%  -37.8%  ⚠
    M                 ███████░░░░░ 58.6%  →  ████████████ 96.4%  +37.8%  ⚠

────────────────────────────────────────────────────────────────────────
  Recommendation
────────────────────────────────────────────────────────────────

## 7. Fairness Metrics — Before Training and After

### 7a. Demographic Parity Gap (no model needed)

Computable from the dataset labels alone. This is the metric that can be logged *before* training to show the pipeline already introduced bias.

In [11]:
# DPG on the BIASED cleaned dataset (post-dropna)
dpg_biased = DemographicParityGap.compute(d2, 'gender', 'subsidy_eligible')
dpg_biased.print_report()

# Log it against the attributed step run
clean_step = STORE.get_steps('clean_data')[0]
STORE.log_metrics(**dpg_biased.to_store_kwargs(
    run_id=clean_step['run_id'], step_name='clean_data'))
print(f'DPG logged: {dpg_biased.primary_value:.4f}')


════════════════════════════════════════════════════════════════════
  Fairness Report: demographic_parity_gap
════════════════════════════════════════════════════════════════════
  Sensitive column: gender
  Outcome column:   subsidy_eligible

  Group                 Rate / Value  Bar
  ──────────────────  ──────────────  ─────────────────────────
  M                            64.2%  ████████████████░░░░░░░░░
  F                            40.0%  ██████████░░░░░░░░░░░░░░░

  Gap:  0.2418  (24.2%)

  Group 'F' has a 24.2% lower selection rate than group 'M' (40.0% vs 64.2%). This exceeds the 4/5ths rule threshold (potential discrimination).

════════════════════════════════════════════════════════════════════

DPG logged: 0.2418


### 7b. Equalized Odds and Predictive Parity (requires model)

After training, we can compute model-level fairness metrics. These require predictions — note how they tell a different story to DPG (Chouldechova tension).

In [12]:
# Predict on the biased test set
X_test = d5[features]
d2_eval = d2.copy()
d2_eval['predicted'] = model_biased.predict(d5[features])

# Equalized Odds
eo = EqualizedOdds.compute(d2_eval, 'gender', 'subsidy_eligible', 'predicted')
eo.print_report()

# Predictive Parity
pp = PredictiveParity.compute(d2_eval, 'gender', 'subsidy_eligible', 'predicted')
pp.print_report()

print('Note: DPG measures selection rate gap (from data).')
print('EO measures TPR/FPR gap (from model behaviour).')
print('PP measures precision gap (from model confidence).')
print('When base rates differ, these cannot all be zero simultaneously (Chouldechova 2017).')


════════════════════════════════════════════════════════════════════
  Fairness Report: equalized_odds_gap
════════════════════════════════════════════════════════════════════
  Sensitive column: gender
  Outcome column:   subsidy_eligible
  Prediction col:   predicted

  Group                 Rate / Value  Bar
  ──────────────────  ──────────────  ─────────────────────────
  M                           100.0%  █████████████████████████
  F                           100.0%  █████████████████████████

  Gap:  0.0000  (0.0%)

  TPR gap: 0.0% ('M': 100.0% vs 'M': 100.0%). FPR gap: 0.0%. EO gap: 0.0%.

  Group                    TPR       FPR
  ──────────────────  ────────  ────────
  F                     100.0%      0.0%
  M                     100.0%      0.0%

  TPR gap: 0.0000   FPR gap: 0.0000

════════════════════════════════════════════════════════════════════


════════════════════════════════════════════════════════════════════
  Fairness Report: predictive_parity_gap
══════════

### 7c. Regression Fairness — Yield Prediction

For the yield prediction sub-problem (continuous output), use `RegressionFairnessAuditor`. It runs four diagnostics including the lazy solution guard.

In [13]:
from sklearn.linear_model import Ridge

reg_features = ['farm_size_ha', 'soil_quality', 'experience_yrs', 'land_title_enc']
X_reg = d5[reg_features]
y_reg = d2['crop_yield_t_ha'].values  # align index

# Quick yield regression model
reg_model = Ridge(alpha=1.0)
reg_model.fit(X_reg, y_reg)

d2_reg = d2.copy()
d2_reg['yield_pred'] = reg_model.predict(X_reg)

auditor = RegressionFairnessAuditor(lazy_threshold=0.05)
reg_report = auditor.audit(d2_reg, 'gender', 'crop_yield_t_ha', 'yield_pred')
reg_report.print_report()


════════════════════════════════════════════════════════════════════════
  Regression Fairness Audit Report
════════════════════════════════════════════════════════════════════════
  Sensitive column: gender
  Outcome column:   crop_yield_t_ha
  Prediction col:   yield_pred

  Group                  N         ME       MAE   Res.Std   Calib.Err   Lazy?
  ────────────────  ──────  ─────────  ────────  ────────  ──────────  ──────
  F                      5    +0.0630    0.2016    0.3151      0.0630      no
  M                    134    -0.0024    0.2760    0.3424      0.0024      no

  ME gap (max pairwise):          0.0654
  MAE gap (max pairwise):         0.0744
  Calibration gap (max error):    0.0630

  ME gap: 0.0654 (worst group: 'F', ME=+0.0630). MAE gap: 0.0744 (worst: 'M'). Calibration gap: 0.0630.

  Warnings:
    • ME gap (0.0654) exceeds 10% of outcome std (0.5677). The model has systematic directional bias: it over-predicts for some groups and under-predicts for others.
   

## 8. Counterfactual Proof (CounterfactualReplayer)

The replayer re-runs the pipeline with a fix at the attributed step and measures the before/after delta. `register_tracked()` reads the metadata directly from `@track` — no double-specification.

In [14]:
def impute_data(df):
    """Fixed: stratified mode imputation preserves demographic balance."""
    df = df.copy()
    for col in ['land_title', 'mobile_number']:
        for g in df['gender'].dropna().unique():
            mask = (df['gender'] == g) & df[col].isna()
            fill = df.loc[df['gender'] == g, col].mode()
            df.loc[mask, col] = fill.iloc[0] if len(fill) > 0 else 'unknown'
    # Impute GPS with group median
    for gps_col in ['latitude', 'longitude']:
        for g in df['gender'].dropna().unique():
            mask = (df['gender'] == g) & df[gps_col].isna()
            df.loc[mask, gps_col] = df.loc[df['gender'] == g, gps_col].median()
    return df


def dpg_bias_fn(df):
    """Bias metric: DPG on subsidy_eligible — computable without a model."""
    f = df[df['gender']=='F']['subsidy_eligible'].mean()
    m = df[df['gender']=='M']['subsidy_eligible'].mean()
    return float(abs(f - m)) if not (np.isnan(f) or np.isnan(m)) else 0.0


print('Replacement function defined.')

Replacement function defined.


In [15]:
# register_tracked() reads name, snapshot, sensitive_cols from @track metadata
replayer = (
    CounterfactualReplayer(store=STORE)
    .register_tracked(load_farm_data)        # reads @track metadata automatically
    .register_tracked(clean_data)            # snapshot=True, sensitive_cols=['gender']
    .register('engineer_features',  engineer_features)
    .register('encode_categoricals', encode_categoricals)
    .register('normalize_features', normalize_features)
)

cf_result = replayer.replay(
    raw_data       = df_raw,
    replace_step   = attributed_step,
    replacement_fn = impute_data,
    sensitive_col  = 'gender',
    bias_metric_fn = dpg_bias_fn,
)
replayer.print_result(cf_result)


════════════════════════════════════════════════════════════════════════
  DataLineageML — Counterfactual Replay Report
════════════════════════════════════════════════════════════════════════
  Replaced step:    clean_data
  Sensitive column: gender

  Row counts at 'clean_data':
    Biased pipeline:      139 rows
    Fixed pipeline:       500 rows
    Rows recovered:      +361  (259.7%)

  'gender' distribution at 'clean_data':
  Value             Input  Biased out   Fixed out   Δ recovered
  ──────────────  ───────  ──────────  ──────────  ────────────
  F                 41.4%  ░░░░░░░░  3.6%  ███░░░░░ 41.4%       +37.8%  ✓
  M                 58.6%  ████████ 96.4%  █████░░░ 58.6%       -37.8%  ✓

  Bias metric comparison:
    Before fix:  0.2418
    After fix:   0.2415
    Reduction:   +0.0003  (+0.1%)

  JSD shift improvement: +0.1681
  (positive = demographic distribution moved closer to input)

────────────────────────────────────────────────────────────────────────
  Verdict:

## 9. Cross-Run Drift Detection (CrossRunComparator)

Simulate multiple pipeline runs with declining demographic representation (e.g. as the data collection process deteriorates over time), then detect the trend.

In [16]:
# Simulate 5 runs with declining F representation at clean_data
# (mimics real-world gradual drift in data collection quality)
drift_store = LineageStore(db_path='oyo_drift_simulation.db')

f_fracs = [0.40, 0.36, 0.31, 0.27, 0.22]  # declining over 5 runs
for i, f_frac in enumerate(f_fracs):
    drift_store.log_snapshot(
        run_id      = f'run-{i:03d}',
        step_name   = 'clean_data',
        position    = 'after',
        row_count   = int(N * (0.5 - i * 0.04)),   # also losing rows over time
        column_count = 4,
        column_names = ['gender', 'land_title', 'farm_size_ha', 'yield_t_ha'],
        null_rates   = {},
        numeric_stats = {},
        categorical_stats = {},
        sensitive_stats = {'gender': {'F': round(f_frac, 3),
                                      'M': round(1 - f_frac, 3)}},
        recorded_at  = f'2026-0{3 + i // 30}-{20 + i:02d}T07:00:00',
    )
    print(f'  Run {i}: F = {f_frac:.1%}  (rows: {int(N*(0.5-i*0.04))})')

print(f'\nSimulated {len(f_fracs)} pipeline runs.')

  Run 0: F = 40.0%  (rows: 250)
  Run 1: F = 36.0%  (rows: 230)
  Run 2: F = 31.0%  (rows: 210)
  Run 3: F = 27.0%  (rows: 190)
  Run 4: F = 22.0%  (rows: 169)

Simulated 5 pipeline runs.


In [17]:
comparator = CrossRunComparator(store=drift_store)

# Compare distributions across all runs
cross_run_report = comparator.compare_step('clean_data', 'gender')
comparator.print_report(cross_run_report)

# Trend analysis
trend = comparator.trend('clean_data', 'gender', group='F')
print(f'Trend for F representation: {trend["direction"].upper()}')
print(f'Slope per run: {trend["slope"]:+.4f}')
print(f'Fractions over time: {[f"{v:.1%}" for v in trend["fractions"]]}')

if trend['direction'] == 'worsening':
    print('\n⚠ Demographic representation is deteriorating across pipeline runs.')
    print('   Investigate data collection process changes.')


════════════════════════════════════════════════════════════════════════
  Cross-Run Comparison: 'clean_data' → 'gender' (after)
════════════════════════════════════════════════════════════════════════
  Runs found: 5   Max drift between runs: 0.0500

   Run             Recorded    Rows         F         M
  ────  ───────────────────  ──────  ────────  ────────
     0  2026-03-20T07:00:00     250    40.0%    60.0% ✓ best
     1  2026-03-21T07:00:00     230    36.0%    64.0%
     2  2026-03-22T07:00:00     210    31.0%    69.0%
     3  2026-03-23T07:00:00     190    27.0%    73.0%
     4  2026-03-24T07:00:00     169    22.0%    78.0% ⚠ worst

  Run-to-run deltas:
    Run 0 → 1:       F:  -4.0%       M:  +4.0%
    Run 1 → 2:       F:  -5.0%       M:  +5.0%
    Run 2 → 3:       F:  -4.0%       M:  +4.0%
    Run 3 → 4:       F:  -5.0%       M:  +5.0%

════════════════════════════════════════════════════════════════════════

Trend for F representation: WORSENING
Slope per run: -0.0450
Frac

## 10. Export Audit Report (generate_report)

Generate a self-contained HTML report — suitable for sharing with regulators, team leads, or PhD supervisors.

In [18]:
report_path = generate_report(
    store                 = STORE,
    output_path           = 'oyo_audit_report.html',
    pipeline_name         = 'oyo_subsidy_pipeline_v1_biased',
    sensitive_col         = 'gender',
    attribution_result    = attribution,
    counterfactual_result = cf_result,
    title                 = 'Oyo State Agricultural Subsidy Pipeline — Fairness Audit',
)

print(f'Report saved → {report_path}')
print('Open with: open oyo_audit_report.html')
print(f'Size: {os.path.getsize(report_path) / 1024:.1f} KB')

Report saved → oyo_audit_report.html
Open with: open oyo_audit_report.html
Size: 24.9 KB


## 11. The Fixed Pipeline

Re-run with stratified imputation. Compare the full audit trail.

In [19]:
dlm.init(db_path='oyo_lineage_fixed.db')
STORE_FIXED = dlm.get_default_store()

@track(
    name='clean_data_fixed',
    tags={'stage': 'preprocessing', 'strategy': 'stratified_imputation'},
    snapshot=True,
    sensitive_cols=['gender'],
)
def clean_data_fixed(df):
    return impute_data(df)

print('Running FIXED pipeline...\n')
with LineageContext(name='oyo_subsidy_pipeline_v2_fixed'):
    f1 = load_farm_data(df_raw)
    f2 = clean_data_fixed(f1)
    f3 = engineer_features(f2)
    f4 = encode_categoricals(f3)
    f5 = normalize_features(f4)
    model_fixed, _ = train_subsidy_model(f5)

print(f'\nRows after clean_data_fixed: {len(f2)} / {len(f1)} ({len(f2)/len(f1):.1%} retained)')

# Detect shifts on fixed pipeline
det_fixed  = ShiftDetector(store=STORE_FIXED)
shifts_fixed = det_fixed.detect()
det_fixed.print_report(shifts_fixed, title='Shift Report — FIXED pipeline')

Running FIXED pipeline...

  CV accuracy: 0.996 ± 0.005

Rows after clean_data_fixed: 500 / 500 (100.0% retained)

════════════════════════════════════════════════════════════════════════
  Shift Report — FIXED pipeline
════════════════════════════════════════════════════════════════════════
  Steps analysed: 1   Signals: 8   HIGH: 0   MEDIUM: 2   LOW: 6
  JSD thresholds: HIGH ≥ 0.020  MEDIUM ≥ 0.005   KS thresholds: HIGH ≥ 0.20  MEDIUM ≥ 0.10

  #    Step                     Column         Test     Stat  Flag          Rows removed
  ──── ──────────────────────── ────────────── ───── ───────  ──────────  ──────────────
  1    clean_data_fixed         latitude       ks     0.1426  △ MEDIUM           0 (0.0%)
  2    clean_data_fixed         longitude      ks     0.1171  △ MEDIUM           0 (0.0%)
  3    clean_data_fixed         gender         jsd    0.0000    LOW              0 (0.0%)
  4    clean_data_fixed         crop_yield_t_ha ks     0.0000    LOW              0 (0.0%)
  5    clean

## 12. Final Summary

In [20]:
# DPG on fixed pipeline
dpg_fixed = DemographicParityGap.compute(f2, 'gender', 'subsidy_eligible')

# Model predictions on full test set
f2_eval = f2.copy()
f2_eval['predicted'] = model_fixed.predict(f5[features])
eo_fixed = EqualizedOdds.compute(f2_eval, 'gender', 'subsidy_eligible', 'predicted')

print('=' * 68)
print('  DataLineageML v0.2 — Complete Bias Attribution Summary')
print('=' * 68)
print()
print(f'  Dataset:            {N} synthetic farm records (40% female-headed)')
print(f'  Attributed step:    {attributed_step}  (confidence {attribution["confidence"]:.0%})')
print(f'  Root mechanism:     dropna() on land_title removes {1-len(d2)/N:.0%} of dataset')
print()
print(f'  ── Demographic Parity Gap (DPG) ─────────────────────────────')
print(f'     Biased pipeline:  {dpg_biased.primary_value:.4f}  ({dpg_biased.primary_value:.1%})')
print(f'     Fixed pipeline:   {dpg_fixed.primary_value:.4f}  ({dpg_fixed.primary_value:.1%})')
reduction = (dpg_biased.primary_value - dpg_fixed.primary_value) / dpg_biased.primary_value * 100
print(f'     Reduction:        {reduction:+.1f}%')
print()
print(f'  ── Equalized Odds Gap ───────────────────────────────────────')
print(f'     Biased pipeline:  {eo.primary_value:.4f}')
print(f'     Fixed pipeline:   {eo_fixed.primary_value:.4f}')
print()
print(f'  ── Counterfactual verdict: {cf_result["verdict"]} ─────────────────────')
print(f'     {cf_result["verdict_detail"]}')
print()
print(f'  Audit report saved → oyo_audit_report.html')
print('=' * 68)

  DataLineageML v0.2 — Complete Bias Attribution Summary

  Dataset:            500 synthetic farm records (40% female-headed)
  Attributed step:    clean_data  (confidence 100%)
  Root mechanism:     dropna() on land_title removes 72% of dataset

  ── Demographic Parity Gap (DPG) ─────────────────────────────
     Biased pipeline:  0.2418  (24.2%)
     Fixed pipeline:   0.2415  (24.2%)
     Reduction:        +0.1%

  ── Equalized Odds Gap ───────────────────────────────────────
     Biased pipeline:  0.0000
     Fixed pipeline:   0.0000

  ── Counterfactual verdict: WEAK ─────────────────────
     Bias metric reduced by only 0.1%. The replacement helps marginally. The attributed step may not be the sole causal source.

  Audit report saved → oyo_audit_report.html


## What Was Built (v0.2 Complete)

| Layer | Module | Status |
|---|---|---|
| Lineage tracking | `@track`, `LineageContext`, `LineageStore` | ✅ v0.1 |
| Snapshot logging | `DataFrameProfiler` | ✅ v0.2 |
| Sensitive col discovery | `discover_sensitive_cols` | ✅ v0.2 |
| Shift detection | `ShiftDetector` (JSD + KS) | ✅ v0.2 |
| Causal attribution | `CausalAttributor` | ✅ v0.2 |
| Fairness metrics | `DemographicParityGap`, `EqualizedOdds`, `PredictiveParity`, `RegressionFairnessAuditor` | ✅ v0.2 |
| Counterfactual proof | `CounterfactualReplayer` | ✅ v0.2 |
| Drift detection | `CrossRunComparator` | ✅ v0.2 |
| Report export | `generate_report` | ✅ v0.2 |

## What Could Be Added (v0.3 and Beyond)

**Genuine research contributions:**
- **Wasserstein distance** as an alternative shift metric — better for continuous distributions and ordinal categories
- **Causal graph discovery** — automatically infer which upstream steps influence each downstream step using structural causal models (Pearl 2009)
- **Multi-step attribution** — currently attributes to a single step; real bias often has multiple contributing causes at different levels
- **Cross-agent provenance** — track demographic shifts through LLM fine-tuning pipelines where data flows across multiple agents
- **Formal verification** of counterfactual claims using do-calculus

**High-value polish:**
- **CLI**: `datalineageml audit pipeline.db --sensitive gender` — makes the tool usable without writing code
- **sklearn Pipeline wrapper** — auto-track every `fit_transform` step
- **Streamlit dashboard** — real-time pipeline monitoring for non-technical users
- **PyTorch DataLoader integration** — track dataset versions fed into training loops

**What was deliberately excluded from v0.2:**
- Cloud backends (the offline-first design is a feature, not a limitation)
- Async step support in the replayer (adds complexity without enabling new use cases)
- Branching pipeline replay (the 80% DataFrame-in/DataFrame-out constraint is intentional)